# Week 6 Lab 04: Evaluating Text Generation with BLEU

**Cordwell Home and Hardware scenario**

**Course:** Model Evaluation Metrics and Experiment Tracking
**Day 2 Theme:** Evaluation metrics for NLP and generated text (BLEU, ROUGE, and beyond)
**Duration:** about 120 minutes
**Runs on:** CPU only. No GPU, no network, no API keys, no Docker. Any laptop works.

## Scenario

Cordwell Home and Hardware is a big-box home improvement retailer with thousands of products and store associates across flooring, paint, appliances, tools, garden, and more.

The Cordwell data science team is piloting **two text-generation systems** that rewrite internal knowledge-base answers into customer-ready responses. Think of this as English to English rewriting: same facts, friendlier delivery. Before either system touches a real customer, the team needs a number that says which one is better.

That number, for this lab, is **BLEU**. You will:

- Build a seeded synthetic corpus of 500 Cordwell knowledge-base answers.
- Simulate the two rewriting systems, System A and System B, with different quality characteristics.
- Compute **corpus-level and sentence-level BLEU** with `sacrebleu` and read every component of the score, not just the headline number.
- Hunt down the rows where BLEU disagrees with your own judgment, and figure out why. One of the systems has a lazy shortcut buried in its behavior, and you are going to catch it with the metric.
- Rebuild BLEU by hand on one example so the formula from the slides stops being a formula and becomes code you wrote.

BLEU was designed for machine translation, but here we use it the way industry often does in practice: as a **surface-similarity metric** between reference answers and system outputs.


## Learning Objectives

By the end of this lab, you should be able to:

1. Explain in your own words what BLEU measures and what its three moving parts are: n-gram precision, clipping, and the brevity penalty.
2. Compute corpus-level BLEU with `sacrebleu` and decompose a score into its per-order precisions and brevity penalty.
3. Compute sentence-level BLEU, describe why it is noisier than corpus-level BLEU, and read a score distribution.
4. Use BLEU deltas to find and diagnose rows where a metric rewards the wrong behavior, including verbatim copying.
5. Implement clipped n-gram precision and the brevity penalty by hand and reconcile your numbers with `sacrebleu`.

## How this lab works

- Cells marked **TODO** are yours. Everything else is provided so your time goes to the evaluation concepts, not plumbing.
- A soft check harness is defined near the top. Run the final `run_checks()` cell any time to see where you stand. Checks report **PASS**, **FAIL**, or **TODO** and never crash the notebook.
- Two hint files ship with this lab. Pick **one tier per task**, not both:
  - `HINTS.md` gives three escalating nudges per task. Start here if you want to work it out yourself.
  - `HINTS_DETAILED.md` shows the working core of each task with line-by-line commentary. Use it when you want to read real code and understand it, then write your own.
- The solution notebook is withheld until after the lab.


## 0. Environment Setup

This lab runs entirely offline on CPU. We use:

- `pandas` for corpus handling.
- `numpy` for numeric work.
- `matplotlib` for two small plots.
- `sacrebleu` for standardized BLEU computation. This is the same implementation used in machine translation research and in the WMT shared tasks, which is exactly why we use it: everyone computing BLEU with sacrebleu on the same data gets the same number.

If you are missing packages, uncomment and run the install cell once, then restart the kernel.


In [ ]:
%pip install -r requirements.txt

In [ ]:
import math
import random
from collections import Counter
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sacrebleu import corpus_bleu, sentence_bleu

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True

pd.set_option("display.max_colwidth", 160)

print("sacrebleu ready. Note: sacrebleu reports BLEU on a 0 to 100 scale, not 0 to 1.")


In [ ]:
# ------------------------------------------------------------------
# Soft check harness. Provided. Do not edit.
# Run run_checks() at any time. Unfinished tasks report TODO, wrong
# answers report FAIL with a message, and nothing here ever crashes.
# ------------------------------------------------------------------

def _need(g, *names):
    """Raise NotImplementedError if any required name is missing."""
    for n in names:
        if n not in g:
            raise NotImplementedError(f"define {n} first")


def _check_task1(g):
    _need(g, "bleu_a", "bleu_b")
    a, b = g["bleu_a"], g["bleu_b"]
    assert hasattr(a, "score") and hasattr(b, "score"), (
        "bleu_a and bleu_b should be the BLEUScore objects returned by corpus_bleu, "
        "not plain floats"
    )
    assert abs(a.score - 67.51) < 0.05, f"bleu_a.score is {a.score:.2f}, expected about 67.51"
    assert abs(b.score - 19.55) < 0.05, f"bleu_b.score is {b.score:.2f}, expected about 19.55"


def _check_task2(g):
    _need(g, "components_df")
    cdf = g["components_df"]
    for col in ["bleu", "p1", "p2", "p3", "p4", "bp"]:
        assert col in cdf.columns, f"components_df is missing column {col!r}"
    for idx in ["System A", "System B"]:
        assert idx in cdf.index, f"components_df is missing row {idx!r}"
    assert abs(cdf.loc["System A", "bp"] - 1.0) < 0.001, "System A brevity penalty should be 1.0"
    assert abs(cdf.loc["System B", "bp"] - 0.293) < 0.005, (
        f"System B brevity penalty is {cdf.loc['System B', 'bp']:.3f}, expected about 0.293"
    )
    assert cdf.loc["System B", "p4"] > cdf.loc["System A", "p4"], (
        "Look again: System B should have HIGHER 4-gram precision than System A. "
        "That surprise is the point of this task."
    )


def _check_task3(g):
    _need(g, "cordwell_df")
    df = g["cordwell_df"]
    if "bleu_A" not in df.columns or "bleu_B" not in df.columns:
        raise NotImplementedError("add bleu_A and bleu_B columns to cordwell_df")
    assert len(df) == 500, "cordwell_df should still have 500 rows"
    assert abs(df["bleu_A"].mean() - 67.46) < 0.2, (
        f"mean of bleu_A is {df['bleu_A'].mean():.2f}, expected about 67.46"
    )
    assert abs(df["bleu_B"].mean() - 20.49) < 0.2, (
        f"mean of bleu_B is {df['bleu_B'].mean():.2f}, expected about 20.49"
    )


def _check_task4(g):
    _need(g, "plot_bleu_histograms", "cordwell_df")
    df = g["cordwell_df"]
    if "bleu_A" not in df.columns:
        raise NotImplementedError("complete Task 3 first so the plot has data")
    result = g["plot_bleu_histograms"](df)
    assert isinstance(result, tuple) and len(result) == 2, (
        "plot_bleu_histograms should return (fig, ax)"
    )
    fig, ax = result
    assert ax.get_legend() is not None, "add a legend so the two systems are labeled"
    assert len(ax.patches) > 10, "expected two histograms drawn on the axes"
    assert ax.get_xlabel() != "", "label the x axis"
    plt.close(fig)


def _check_task5(g):
    _need(g, "b_wins_df", "n_passthrough_wins")
    bw = g["b_wins_df"]
    assert len(bw) == 33, f"b_wins_df has {len(bw)} rows, expected 33"
    assert g["n_passthrough_wins"] == 33, (
        f"n_passthrough_wins is {g['n_passthrough_wins']}, expected 33. "
        "Compare candidate_B to reference with a simple equality check."
    )


def _check_task6(g):
    _need(g, "a_wins_df")
    aw = g["a_wins_df"]
    assert len(aw) == 465, f"a_wins_df has {len(aw)} rows, expected 465"


def _check_task7(g):
    _need(g, "ngrams", "modified_precision")
    out = g["ngrams"](["a", "b", "c"], 2)
    assert list(out) == [("a", "b"), ("b", "c")], (
        f"ngrams(['a','b','c'], 2) returned {out!r}, expected [('a','b'), ('b','c')]"
    )
    p = g["modified_precision"](
        "the the the the the the the".split(),
        "the cat is on the mat".split(),
        1,
    )
    assert abs(p - 2 / 7) < 1e-9, (
        f"clipping demo precision is {p:.4f}, expected 2/7 = 0.2857. "
        "Remember to clip each candidate n-gram count by its reference count."
    )


def _check_task8(g):
    _need(g, "brevity_penalty", "manual_p1", "manual_p2", "manual_bp", "manual_bleu2")
    bp_fn = g["brevity_penalty"]
    assert bp_fn(64, 59) == 1.0, "brevity_penalty(64, 59) should be exactly 1.0"
    assert abs(bp_fn(27, 59) - 0.30566) < 1e-3, (
        f"brevity_penalty(27, 59) is {bp_fn(27, 59):.5f}, expected about 0.30566"
    )
    assert abs(g["manual_bp"] - 1.0) < 1e-9, "manual_bp for candidate A should be 1.0"
    assert abs(g["manual_bleu2"] - 77.76) < 0.1, (
        f"manual_bleu2 is {g['manual_bleu2']:.2f}, expected about 77.76 on the 0 to 100 scale"
    )


CHECKS = [
    ("Task 1: corpus BLEU", _check_task1),
    ("Task 2: score components", _check_task2),
    ("Task 3: sentence BLEU columns", _check_task3),
    ("Task 4: histogram function", _check_task4),
    ("Task 5: where B beats A", _check_task5),
    ("Task 6: where A crushes B", _check_task6),
    ("Task 7: clipped precision", _check_task7),
    ("Task 8: brevity penalty and BLEU-2", _check_task8),
]


def run_checks():
    g = globals()
    passed = 0
    print(f"{'Check':<38} {'Status':<8} Detail")
    print("-" * 90)
    for name, fn in CHECKS:
        try:
            fn(g)
            status, detail = "PASS", ""
            passed += 1
        except NotImplementedError as e:
            status, detail = "TODO", str(e)
        except AssertionError as e:
            status, detail = "FAIL", str(e)
        except Exception as e:
            status, detail = "FAIL", f"{type(e).__name__}: {e}"
        print(f"{name:<38} {status:<8} {detail}")
    print("-" * 90)
    print(f"PASSED {passed} of {len(CHECKS)}")
    return passed


print("Check harness loaded. Call run_checks() at any time.")


## 1. Conceptual Warm-Up: From Exact Match to BLEU

Recall from the slides that we can compare generated text to a reference at three levels:

1. **Exact match:** character or token identity. Brittle, but trivially cheap.
2. **Surface similarity:** n-gram overlap. This is BLEU and ROUGE territory.
3. **Semantic equivalence:** embeddings, BERTScore, human judgment.

This lab lives at level 2. BLEU combines three ideas, and every task below touches at least one of them:

- **N-gram precision:** what fraction of the candidate's n-grams also appear in the reference. Precision, not recall: it asks "of what the system wrote, how much is supported," not "how much of the reference did it cover."
- **Clipping:** a candidate cannot get credit for the same reference n-gram more times than it appears in the reference. This kills the classic degenerate output that repeats one common word.
- **Brevity penalty (BP):** because BLEU is precision-based, an ultra-short candidate could look deceptively precise. BP multiplies the score down when the candidate is shorter than the reference.

### Exercise 1.1: Quick mental check (no code)

Imagine these Cordwell-style answers:

- **Reference:** Our flooring department is usually open from 7am to 9pm on weekdays, but hours can vary by location. Check the store page or call ahead for the most accurate schedule.
- **Candidate A:** The flooring aisle is typically open from 7:00 in the morning until 9:00 at night on weekdays, though exact hours depend on the store. It is best to confirm online or by phone.
- **Candidate B:** Store hours are online. Times can change.

Before running anything: which candidate will BLEU score higher, and why? Which would you prefer as a customer? Keep your answer in mind; Section 5 revisits this tension at scale.


### Worked example: every tool you need for this lab, on a toy pair

The cell below is provided and fully worked. It shows, on tiny inputs, every API and formula pattern the tasks ask you to use:

- `collections.Counter` for counting n-grams (used in Task 7).
- Clipped unigram precision computed by hand on the famous degenerate example from the original BLEU paper.
- `corpus_bleu(hypotheses, [references])` from sacrebleu, and how to read `.score`, `.precisions`, `.bp`, `.sys_len`, and `.ref_len` off the result (used in Tasks 1 and 2).

Two argument-order facts worth tattooing somewhere: sacrebleu takes the **candidates first, references second**, and the references go in as a **list of reference lists**, because BLEU supports multiple references per segment. We use one reference throughout this lab, so you will always see the pattern `[references]`.

Read this cell closely. It is the pattern library for everything you write today.


In [ ]:
# ---------------------------------------------------------------
# Worked example 1: clipping, by hand, with collections.Counter
# ---------------------------------------------------------------
# The degenerate candidate from the original BLEU paper (Papineni et al., 2002):
candidate = "the the the the the the the".split()
reference = "the cat is on the mat".split()

# Count unigrams on each side. Counter maps each item to how many times it appears.
cand_counts = Counter(candidate)          # {'the': 7}
ref_counts = Counter(reference)           # {'the': 2, 'cat': 1, ...}
print("candidate counts:", dict(cand_counts))
print("reference counts:", dict(ref_counts))

# Unclipped precision would be 7/7 = 1.0, a perfect score for garbage.
# Clipping caps each candidate n-gram count at its reference count.
clipped = sum(min(count, ref_counts[gram]) for gram, count in cand_counts.items())
total = sum(cand_counts.values())
print(f"clipped unigram precision: {clipped}/{total} = {clipped / total:.4f}")

# ---------------------------------------------------------------
# Worked example 2: corpus_bleu on a two-sentence toy corpus
# ---------------------------------------------------------------
toy_refs = ["The cat sat on the mat.", "Paint dries in about four hours."]
toy_hyps = ["The cat sat on the mat.", "Paint usually dries in four hours."]

# Candidates FIRST, then a LIST of reference lists.
toy_bleu = corpus_bleu(toy_hyps, [toy_refs])

print()
print("full result:", toy_bleu)
print(f"score      : {toy_bleu.score:.2f}   (0 to 100 scale)")
print(f"precisions : {[round(p, 1) for p in toy_bleu.precisions]}   (1-gram through 4-gram)")
print(f"bp         : {toy_bleu.bp:.3f}")
print(f"lengths    : sys_len={toy_bleu.sys_len}, ref_len={toy_bleu.ref_len}")


## 2. Building the Cordwell Corpus (provided)

The next four cells build a **seeded, deterministic** corpus of 500 knowledge-base answers. Every run of this notebook produces the exact same corpus, which is why the check harness can verify your numbers precisely. This is also lab hygiene worth copying into your own eval work: an evaluation you cannot reproduce is an anecdote, not an evaluation.

Each row has:

- A **scenario type** (store hours, installation, returns, delivery, and so on).
- A detailed **reference answer**, the ideal Cordwell associate response.
- **candidate_A** from System A: a decent paraphraser. It swaps synonyms, rewrites common phrases, occasionally flips clauses and reorders sentences. Its output is usually faithful but sometimes clunky, like a real mid-tier rewriting model.
- **candidate_B** from System B: a lower-effort system. It usually keeps only the first sentence or two, truncates them, and pastes on a generic closer. It also has one more behavior that nobody documented. You will find it in Section 5 using nothing but the metric.

This is provided plumbing. Skim it so you know how the sausage is made, but do not spend lab time here; your work starts in Section 3.


In [ ]:
# Corpus vocabulary and rewrite tables. Provided.

DEPARTMENTS = [
    "flooring", "paint", "lighting", "appliances", "plumbing",
    "kitchen cabinets", "garden", "tools", "lumber", "storage",
]

PRODUCTS = [
    "laminate flooring", "vinyl plank", "ceramic tile", "ceiling fan",
    "front-load washer", "gas dryer", "programmable thermostat",
    "cordless drill", "decking boards", "bathroom vanity",
    "LED shop light", "exterior paint", "interior primer",
    "mulch", "potted shrubs", "storm door", "garage storage rack",
]

CITIES = [
    "Columbus", "Atlanta", "Phoenix", "Seattle", "Chicago",
    "Denver", "Austin", "Charlotte", "Orlando", "Cleveland",
]

WEEKEND_DAYS = ["Saturday", "Sunday"]

SCENARIO_TYPES = [
    "store_hours",
    "installation_services",
    "inventory_and_stock",
    "delivery_and_pickup",
    "returns_and_exchanges",
    "warranty_and_protection",
    "product_compatibility",
    "special_order",
    "pricing_and_promotions",
]

SYNONYM_MAP = {
    "department": ["section", "area", "aisle"],
    "store": ["location", "branch"],
    "recommend": ["suggest", "advise"],
    "check": ["look up", "review"],
    "call": ["phone", "reach out to"],
    "website": ["site", "online store"],
    "customer": ["shopper", "guest"],
    "associate": ["team member", "sales associate"],
    "installation": ["install", "setup"],
    "delivery": ["home delivery", "truck delivery"],
    "pickup": ["pick up", "collection"],
    "pricing": ["cost", "pricing details"],
    "typically": ["usually", "generally"],
    "items": ["products", "merchandise"],
    "vary": ["differ", "change"],
    "confirm": ["verify", "double-check"],
    "exact": ["precise", "specific"],
    "details": ["specifics", "information"],
    "options": ["choices", "selections"],
    "available": ["offered", "on hand"],
    "schedule": ["book", "arrange"],
    "free": ["complimentary", "no-cost"],
    "purchase": ["order", "buy"],
    "review": ["go over", "read through"],
    "current": ["latest", "up-to-date"],
    "accurate": ["reliable", "precise"],
    "professional": ["expert", "qualified"],
    "many": ["numerous", "a range of"],
    "most": ["the majority of", "nearly all"],
    "always": ["consistently"],
    "different": ["distinct", "not the same"],
}

PHRASE_MAP = {
    "is typically open": ["usually operates", "is generally open"],
    "we always recommend": ["we suggest", "it is a good idea"],
    "can vary by": ["may differ by", "often depends on the"],
    "the best option is": ["the easiest way is", "your best bet is"],
    "you can": ["you are able to", "feel free to"],
    "will review": ["will go over", "will look at"],
    "may cover": ["can include coverage for", "may include"],
    "it is important to": ["make sure you", "be sure to"],
    "help you": ["assist you to", "work with you to"],
}

OPENERS = ["Please note that", "Keep in mind that", "As a quick reminder,", "Just so you know,"]

GENERIC_CLOSERS = [
    "You can always check the Cordwell website or app for the most current details.",
    "For the latest information, look up your local store online or give them a quick call.",
    "Exact options and availability may vary by store and by region.",
]


In [ ]:
# Reference answer templates. Provided.

def _random_time_range(rng: random.Random) -> Tuple[str, str]:
    start_hour = rng.choice(range(6, 10))
    end_hour = rng.choice(range(6, 10))
    return f"{start_hour}am", f"{end_hour}pm"


def build_reference_answer(scenario: str, rng: random.Random) -> str:
    dept = rng.choice(DEPARTMENTS)
    product = rng.choice(PRODUCTS)
    city = rng.choice(CITIES)
    open_time, close_time = _random_time_range(rng)
    weekend_day = rng.choice(WEEKEND_DAYS)

    if scenario == "store_hours":
        return (
            f"At our Cordwell Home and Hardware store in {city}, the {dept} department is typically open "
            f"from {open_time} to {close_time} on weekdays. Weekend hours can be different, especially on "
            f"{weekend_day}. Holiday schedules may vary, so we always recommend checking the store page "
            "online or calling ahead to confirm exact opening and closing times for your visit."
        )
    if scenario == "installation_services":
        return (
            f"Cordwell offers professional installation services for many {dept} products, including {product}. "
            "Availability and pricing can vary by store and by project size. To get an accurate quote, "
            "schedule a free in-home or virtual consultation through our website or by calling the "
            "installation desk. A licensed and insured installer will review your space, confirm "
            "measurements, and walk you through next steps."
        )
    if scenario == "inventory_and_stock":
        return (
            f"For real-time inventory on items like {product} in our {dept} department, the best option is "
            "to check the product page on the Cordwell website or app. You can view how many units are in "
            "stock at your preferred store and reserve items for pickup. Because stock levels change "
            "throughout the day, we recommend calling the store if you are making a long trip."
        )
    if scenario == "delivery_and_pickup":
        return (
            "Most large or bulky items at Cordwell, such as appliances, lumber, and grills, are eligible "
            f"for home delivery or curbside pickup. For products from the {dept} department, you can choose "
            "a delivery window during checkout or select free store pickup when available. Delivery fees "
            "depend on distance, order size, and current promotions. We always recommend reviewing the "
            "delivery details on the checkout page before you place your order."
        )
    if scenario == "returns_and_exchanges":
        return (
            f"The standard Cordwell return policy allows most new, unopened items from our {dept} department "
            "to be returned within 90 days of purchase with a receipt. Some items, such as special orders, "
            "cut materials, or clearance merchandise, may have different rules. Returns can usually be "
            "processed at any store location, but bringing your original receipt and the card used for "
            "payment will speed things up."
        )
    if scenario == "warranty_and_protection":
        return (
            f"Many {product} items at Cordwell come with a manufacturer warranty, and you can often add an "
            "extended protection plan at checkout. These plans may cover mechanical failures or certain "
            "types of accidental damage after the manufacturer warranty ends. Coverage, term length, and "
            "pricing vary by product, so always review the plan brochure or product page details before deciding."
        )
    if scenario == "product_compatibility":
        return (
            f"When choosing {product}, it is important to confirm compatibility with your existing setup. "
            "Check the product specifications for voltage, size, and any required accessories. If you are "
            "unsure, you can bring photos and measurements into the store, and a Cordwell associate from "
            f"the {dept} department can help you select components that work together."
        )
    if scenario == "special_order":
        return (
            f"If you do not see the exact {product} you want on the shelf, many items in the {dept} "
            "department can be ordered through the Cordwell special-order program. Lead times can range "
            "from a few days to several weeks depending on the vendor and customization options. A store "
            "associate can help you review finish options, sizes, and pricing, and will provide an "
            "estimated arrival date before you place the order."
        )
    return (
        f"Prices for items like {product} can change during promotions, seasonal sales, and clearance "
        "events. The most accurate pricing will always appear on the Cordwell product page and at the "
        "shelf tag in-store. If you recently purchased an item that goes on sale, your store may offer a "
        "price adjustment within a short window; policies vary by location, so check with customer "
        "service for details."
    )


In [ ]:
# System A: a decent but imperfect paraphraser. Provided.

def _lower_first(s: str) -> str:
    return s[0].lower() + s[1:] if s else s


def _upper_first(s: str) -> str:
    return s[0].upper() + s[1:] if s else s


def apply_phrases(text: str, rng: random.Random, prob: float = 0.6) -> str:
    """Rewrite common multi-word phrases, each with probability prob."""
    for phrase, alts in PHRASE_MAP.items():
        if phrase in text and rng.random() < prob:
            text = text.replace(phrase, rng.choice(alts), 1)
    return text


def apply_synonyms(text: str, rng: random.Random, prob: float = 0.45) -> str:
    """Swap individual words for synonyms, each with probability prob."""
    out = []
    for tok in text.split():
        key = tok.lower().strip(",.!?;")
        if key in SYNONYM_MAP and rng.random() < prob:
            rep = rng.choice(SYNONYM_MAP[key])
            if tok[0].isupper():
                rep = _upper_first(rep)
            punct = "" if tok[-1].isalnum() else tok[-1]
            out.append(rep + punct)
        else:
            out.append(tok)
    return " ".join(out)


def restructure_sentence(s: str, rng: random.Random) -> str:
    """Occasionally flip clause order around the first comma or prepend an opener."""
    if "," in s and rng.random() < 0.35:
        head, tail = s.split(",", 1)
        s = _upper_first(tail.strip()) + ", " + _lower_first(head.strip())
    if rng.random() < 0.30:
        s = rng.choice(OPENERS) + " " + _lower_first(s)
    return s


def simulate_system_a(reference: str, rng: random.Random) -> str:
    """Phrase rewrites, then synonym swaps, then sentence-level restructuring."""
    text = apply_phrases(reference, rng)
    text = apply_synonyms(text, rng)
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    sentences = [restructure_sentence(s, rng) for s in sentences]
    if len(sentences) > 2 and rng.random() < 0.6:
        last = sentences.pop()
        idx = rng.randint(1, len(sentences))
        sentences.insert(idx, last)
    return ". ".join(sentences) + "."


In [ ]:
# System B and the corpus generator. Provided.

def simulate_system_b(reference: str, rng: random.Random) -> str:
    """A lower-effort rewriter: truncate, drop detail, add a generic closer."""
    # An undocumented shortcut lives in this function. Section 5 will expose it.
    if rng.random() < 0.06:
        return reference

    sentences = [s.strip() for s in reference.split(".") if s.strip()]
    if not sentences:
        return reference

    keep_n = 1 if len(sentences) == 1 else rng.choice([1, 1, 2])
    kept = sentences[:keep_n]

    short_kept = []
    for s in kept:
        words = s.split()
        cutoff = max(6, int(len(words) * rng.uniform(0.5, 0.75)))
        short_kept.append(" ".join(words[:cutoff]))

    closer = ""
    if rng.random() < 0.7:
        closer = " " + rng.choice(GENERIC_CLOSERS)

    return ". ".join(short_kept) + "." + closer


def generate_cordwell_corpus(n: int = 500, seed: int = 42) -> pd.DataFrame:
    """Deterministic corpus: a dedicated RNG instance seeded once, used everywhere."""
    rng = random.Random(seed)
    rows = []
    for i in range(n):
        scenario = rng.choice(SCENARIO_TYPES)
        reference = build_reference_answer(scenario, rng)
        rows.append(
            {
                "id": i,
                "scenario": scenario,
                "reference": reference,
                "candidate_A": simulate_system_a(reference, rng),
                "candidate_B": simulate_system_b(reference, rng),
            }
        )
    return pd.DataFrame(rows)


cordwell_df = generate_cordwell_corpus(n=500, seed=42)
print(f"Corpus shape: {cordwell_df.shape}")
cordwell_df.head(3)


Take a minute to eyeball a few rows and convince yourself that:

- The **reference** answers are detailed and realistic for a Cordwell retail context.
- **candidate_A** usually preserves the content with different wording, and is occasionally clunky. That clunkiness is deliberate: real paraphrase systems garble grammar sometimes, and part of this lab is asking whether BLEU notices.
- **candidate_B** is usually shorter and more generic, and often drops details a customer would need.


In [ ]:
cordwell_df.sample(3, random_state=7)[["scenario", "reference", "candidate_A", "candidate_B"]]


## 3. Corpus-Level BLEU: System A vs. System B

Time to put a number on each system. Corpus-level BLEU pools the n-gram statistics across all 500 rows before computing the score, which is how BLEU was designed to be used and how it correlates best with human judgment.

### Task 1: Compute corpus BLEU for both systems

Compute corpus-level BLEU for System A and System B against the references, using the `corpus_bleu` pattern from the worked example in Section 1.

**Contract**

- Extract the three text columns from `cordwell_df` as plain Python lists.
- Store the full `BLEUScore` result objects in variables named `bleu_a` and `bleu_b`. Keep the objects themselves, not just `.score`; Task 2 needs their internals.
- Print both scores with two decimal places.

**Worked target output** (your numbers should match exactly, because the corpus is seeded):

```text
System A corpus BLEU: 67.51
System B corpus BLEU: 19.55
```


In [ ]:
# TODO Task 1: corpus-level BLEU for System A and System B
#
# Contract:
#   - references, candidates_a, candidates_b: lists pulled from cordwell_df
#   - bleu_a, bleu_b: BLEUScore objects from corpus_bleu
#   - print both scores with two decimals (see worked target output above)

# YOUR CODE HERE


### Task 2: Decompose the scores

A single BLEU number hides the story. The `BLEUScore` object carries the parts: per-order n-gram precisions, the brevity penalty, and the token lengths. Put them side by side and something genuinely surprising falls out of this comparison. Find it before the checkpoint names it.

**Contract**

- Build a DataFrame named `components_df` with index values `"System A"` and `"System B"` and columns `bleu`, `p1`, `p2`, `p3`, `p4`, `bp`, `sys_len`, `ref_len`, filled from `bleu_a` and `bleu_b`.
- `p1` through `p4` come from the `.precisions` list, `bp` from `.bp`, lengths from `.sys_len` and `.ref_len`.
- Display the DataFrame rounded to 3 decimals.

**Worked target output**

```text
           bleu      p1      p2      p3      p4     bp  sys_len  ref_len
System A  67.508  82.343  71.426  63.191  55.885  1.000    37939    34861
System B  19.548  81.374  65.543  62.138  59.403  0.293    15661    34861
```


In [ ]:
# TODO Task 2: build components_df from bleu_a and bleu_b
#
# Contract:
#   - components_df: DataFrame, index ["System A", "System B"],
#     columns ["bleu", "p1", "p2", "p3", "p4", "bp", "sys_len", "ref_len"]
#   - values come from .score, .precisions, .bp, .sys_len, .ref_len
#   - display it rounded to 3 decimals

# YOUR CODE HERE


### Checkpoint 3.1: Read the components like an engineer

Discuss with your group, referencing `components_df`:

1. System B's BLEU is less than a third of System A's. But look at the four precision columns. Whose **4-gram precision** is higher? Sit with that for a second.
2. So if System B's n-grams are at least as "correct" as System A's, what single component is responsible for almost all of B's losses? What behavior of System B does that component exist to punish?
3. System A's `sys_len` is about 9 percent longer than `ref_len`, and its BP is exactly 1.0. What does that tell you about which direction the brevity penalty operates in? Is there any penalty for being too long?
4. A teammate looks only at headline BLEU and says "System B writes bad n-grams." Correct them in one sentence using the table.

The punchline to carry forward: **BLEU components are diagnostic**. The headline number says who won; the components say why.


## 4. Sentence-Level BLEU and Score Distributions

Corpus BLEU gives one number for 500 rows. Now compute BLEU per row, which is noisier but lets us see the distribution and hunt for individual rows where the metric behaves strangely.

One implementation note from the worked example still applies: `sentence_bleu(candidate, [reference])` takes the candidate first and a plain list of reference strings second. It also defaults to smoothed, effective-order BLEU, which exists precisely because raw BLEU-4 collapses to zero on short sentences that have no matching 4-grams.

### Task 3: Sentence-level BLEU columns

**Contract**

- Add two new float columns to `cordwell_df`: `bleu_A` (candidate_A vs. reference) and `bleu_B` (candidate_B vs. reference), each holding `sentence_bleu(...).score` per row.
- A list comprehension over `zip(...)` of the columns is plenty. Expect this to take a second or two for 1000 scores.

**Worked target output** (from the provided summary cell that follows; check the harness for exact tolerances):

```text
       bleu_A  bleu_B
count  500.00  500.00
mean    67.46   20.49
std      9.76   23.57
min     39.07    0.01
25%     60.08    6.31
50%     67.57   13.84
75%     73.46   23.35
max     94.93  100.00
```


In [ ]:
# TODO Task 3: add sentence-level BLEU columns bleu_A and bleu_B to cordwell_df
#
# Contract:
#   - cordwell_df["bleu_A"]: sentence_bleu score of candidate_A vs reference, per row
#   - cordwell_df["bleu_B"]: sentence_bleu score of candidate_B vs reference, per row
#   - values are floats on the 0 to 100 scale (use .score)

# YOUR CODE HERE


In [ ]:
# Provided: summary statistics for your new columns.
if "bleu_A" in cordwell_df.columns and "bleu_B" in cordwell_df.columns:
    display(cordwell_df[["bleu_A", "bleu_B"]].describe().round(2))
else:
    print("Complete Task 3 first, then re-run this cell.")


### Task 4: Visualize the two distributions

Numbers in a `describe()` table are useful; shapes are memorable. Plot both sentence-level distributions on one axes.

**Contract**

- Define a function `plot_bleu_histograms(df)` that:
  - creates a matplotlib figure and axes with `plt.subplots()`,
  - draws histograms of `df["bleu_A"]` and `df["bleu_B"]` on the same axes, each with a label, `alpha=0.6`, and around 30 bins,
  - labels the x axis (sentence-level BLEU) and y axis (number of rows), adds a title and a legend,
  - **returns `(fig, ax)`**.
- Call it once on `cordwell_df` below the definition.

**Worked target output**: one chart with two overlapping histograms. System A forms a bell-ish cluster roughly between 40 and 95. System B piles up between 0 and 40 with a long tail, plus one strange, isolated spike sitting all the way out at 100. If your spike is missing, look at your bin count. That spike is your first clue for Section 5.


In [ ]:
# TODO Task 4: histogram of sentence-level BLEU for both systems

def plot_bleu_histograms(df: pd.DataFrame):
    """Overlaid histograms of df['bleu_A'] and df['bleu_B'].

    Returns:
        (fig, ax): the matplotlib figure and axes.
    """
    raise NotImplementedError("Task 4 not implemented yet")


# Call it here once implemented:
# fig, ax = plot_bleu_histograms(cordwell_df)


### Checkpoint 4.1: Distribution analysis

1. The corpus scores were 67.51 and 19.55. The sentence-level means are 67.46 and 20.49. Close, but not identical, and they never will be exactly equal. Why not? Think about what gets pooled at the corpus level versus averaged at the sentence level, and about smoothing.
2. Which system has the wider spread? What does a standard deviation of 23.6 versus 9.8 tell you about how much you should trust any single sentence-level score?
3. There is a small, isolated spike in System B's distribution at exactly 100. What kind of candidate scores a perfect 100 against its reference? Write down your hypothesis before Section 5; you are about to test it.


## 5. When BLEU Disagrees with Your Intuition

One of the most valuable evaluation skills is **debugging the metric itself**. A metric is a proxy, and every proxy can be gamed, sometimes by accident. In this section you will find the rows where System B, the obviously worse system, outscores System A, and diagnose exactly why.

### Task 5: Find and diagnose the rows where B beats A

**Contract**

- Create a column `delta_B_minus_A` on `cordwell_df` equal to `bleu_B - bleu_A`.
- Build `b_wins_df`: the rows where `delta_B_minus_A >= 5.0`, sorted by `delta_B_minus_A` descending.
- Display the top 5 rows with columns `scenario`, `reference`, `candidate_B`, `bleu_A`, `bleu_B`, `delta_B_minus_A`. Read a couple of the `candidate_B` texts carefully against their references.
- Then quantify what you see: compute `n_passthrough_wins`, the number of rows in `b_wins_df` where `candidate_B` is **character-for-character identical** to `reference`, and print it alongside `len(b_wins_df)`.

**Worked target output** (final print line):

```text
B wins by 5+ BLEU points on 33 rows; 33 of them are verbatim copies of the reference.
```


In [ ]:
# TODO Task 5: rows where System B outscores System A, and why
#
# Contract:
#   - cordwell_df["delta_B_minus_A"] = bleu_B - bleu_A
#   - b_wins_df: rows with delta_B_minus_A >= 5.0, sorted descending by delta
#   - display top 5 with scenario, reference, candidate_B, bleu_A, bleu_B, delta_B_minus_A
#   - n_passthrough_wins: count of b_wins_df rows where candidate_B == reference
#   - print the summary line (see worked target output)

# YOUR CODE HERE


**What you just caught.** Every single row where System B "wins" is a row where it silently returned the reference untouched. About 6 percent of the time, System B does no rewriting at all: it copies its input straight through. Against a reference identical to that input, a verbatim copy scores a perfect 100.

Now connect this to the business goal. The whole point of these systems is to **rewrite** internal answers into customer-ready language. A passthrough does zero useful work, yet BLEU hands it the maximum possible score, because BLEU measures similarity to the reference and nothing else. The metric is not wrong; it is answering a narrower question than the one the business is asking. This is the single most important lesson of the day: **know exactly what question your metric answers, and check whether that is the question you meant to ask.** In production you would pair BLEU with a diversity or edit-distance guard, or a task-specific check that the output actually differs from the input in the intended way.


### Task 6: Where System A rightly dominates

For contrast, collect the rows where System A beats System B by a wide margin. This is the metric working as intended.

**Contract**

- Create `delta_A_minus_B` on `cordwell_df` equal to `bleu_A - bleu_B`.
- Build `a_wins_df`: rows where `delta_A_minus_B >= 15.0`, sorted descending.
- Print how many rows qualify, and display the top 3 with `scenario`, `reference`, `candidate_A`, `candidate_B`, `bleu_A`, `bleu_B`.

**Worked target output** (first line):

```text
A wins by 15+ BLEU points on 465 of 500 rows.
```


In [ ]:
# TODO Task 6: rows where System A beats System B by at least 15 points
#
# Contract:
#   - cordwell_df["delta_A_minus_B"] = bleu_A - bleu_B
#   - a_wins_df: rows with delta_A_minus_B >= 15.0, sorted descending by delta
#   - print the count (see worked target output), then display the top 3 rows
#     with scenario, reference, candidate_A, candidate_B, bleu_A, bleu_B

# YOUR CODE HERE


### Checkpoint 5.2: Interpreting the gaps

For two or three rows from `a_wins_df`:

1. Do you agree that candidate_A is genuinely better than candidate_B for a Cordwell customer? Usually yes, which is BLEU working: more preserved content means more n-gram overlap means a higher score.
2. Now find a row where candidate_A is grammatically clunky (the clause flips produce some awkward sentences). Did BLEU dock it much? What does that tell you about BLEU's sensitivity to fluency when local n-grams survive?
3. List two aspects of answer quality that matter to a customer but that nothing in this section measured. Where would you get those signals in production?


## 6. Manual BLEU: Build the Formula Yourself

You have used BLEU as a black box long enough. Now rebuild its two core mechanisms, clipped n-gram precision and the brevity penalty, and reconcile your numbers with sacrebleu on one fixed example.

The provided cell below selects one corpus row deterministically and shows both candidates. Candidate A is longer than its reference; candidate B is much shorter. That contrast is deliberate: it will exercise **both branches** of the brevity penalty.


In [ ]:
# Provided: a fixed example row for the manual calculation.
example_row = cordwell_df.sample(1, random_state=123).iloc[0]

print(f"Row id: {example_row['id']}   Scenario: {example_row['scenario']}")
print()
print("REFERENCE:")
print(example_row["reference"])
print()
print("CANDIDATE A:")
print(example_row["candidate_A"])
print()
print("CANDIDATE B:")
print(example_row["candidate_B"])


### Task 7: N-grams and clipped precision

Implement the two helper functions, then apply them to candidate A of the example row. The clipping pattern is exactly the one from the worked example in Section 1; the only new part is generalizing from unigrams to n-grams.

**Contract**

- `ngrams(tokens, n)` returns the list of n-gram **tuples** in order. Example: `ngrams(["a", "b", "c"], 2)` returns `[("a", "b"), ("b", "c")]`.
- `modified_precision(candidate_tokens, reference_tokens, n)` counts n-grams on both sides with `Counter`, clips each candidate n-gram count by its reference count, and returns clipped matches divided by total candidate n-grams, as a float between 0 and 1. Guard against an empty candidate by returning 0.0.
- Tokenize the example row's candidate A and reference by simple whitespace `split()`, compute `manual_p1` (unigram) and `manual_p2` (bigram), and print both to 4 decimals.

**Worked target output**

```text
manual p1 (unigram precision): 0.8281
manual p2 (bigram precision) : 0.7302
```


In [ ]:
# TODO Task 7: n-gram extraction and clipped (modified) precision

def ngrams(tokens: List[str], n: int) -> List[tuple]:
    """Return the list of n-gram tuples of tokens, in order."""
    raise NotImplementedError("Task 7 not implemented yet")


def modified_precision(candidate_tokens: List[str], reference_tokens: List[str], n: int) -> float:
    """Clipped n-gram precision: clipped matches / total candidate n-grams."""
    raise NotImplementedError("Task 7 not implemented yet")


# Once implemented:
#   - tokenize example_row["candidate_A"] and example_row["reference"] with .split()
#   - manual_p1 = modified_precision(cand_tokens, ref_tokens, 1)
#   - manual_p2 = modified_precision(cand_tokens, ref_tokens, 2)
#   - print both to 4 decimals (see worked target output)


### Task 8: Brevity penalty, BLEU-2, and reconciliation

Now assemble a BLEU-2 score (geometric mean of unigram and bigram precision, times the brevity penalty) and compare it with sacrebleu on both candidates.

The brevity penalty formula, exactly as sacrebleu implements it:

- If the candidate length is greater than or equal to the reference length, BP is 1.0. There is no penalty for being long; overly long candidates already pay through lower precision.
- Otherwise BP is `exp(1 - ref_len / cand_len)`, which decays toward 0 as the candidate shrinks.

**Contract**

- `brevity_penalty(c_len, r_len)` implements the formula above on token counts.
- For **candidate A** of the example row, using your Task 7 tokens and precisions: compute `manual_bp`, then `manual_bleu2 = manual_bp * exp(0.5 * (log(p1) + log(p2))) * 100`, on the 0 to 100 scale.
- Repeat the same manual calculation inline for **candidate B** of the example row (short candidate, so BP kicks in).
- Compare both against `sentence_bleu(candidate, [reference])` and print everything.

**Worked target output**

```text
CANDIDATE A: c_len=64 r_len=59
  manual BP: 1.0000   manual BLEU-2: 77.76
  sacrebleu sentence BLEU-4: 71.60 (bp=1.000)

CANDIDATE B: c_len=27 r_len=59
  manual BP: 0.3057   manual BLEU-2: 11.99
  sacrebleu sentence BLEU-4: 12.49 (bp=0.323)
```

Notice the reconciliation is close but not exact, and for candidate B even the two BP values differ slightly. Checkpoint 6.1 asks you why.


In [ ]:
# TODO Task 8: brevity penalty, manual BLEU-2, and comparison with sacrebleu

def brevity_penalty(c_len: int, r_len: int) -> float:
    """BP = 1.0 when the candidate is at least as long as the reference,
    otherwise exp(1 - r_len / c_len)."""
    raise NotImplementedError("Task 8 not implemented yet")


# Once implemented, for the example row:
#   Candidate A (reuse cand_tokens, ref_tokens, manual_p1, manual_p2 from Task 7):
#     - manual_bp = brevity_penalty(len(cand_tokens), len(ref_tokens))
#     - manual_bleu2 = manual_bp * math.exp(0.5 * (math.log(manual_p1) + math.log(manual_p2))) * 100
#   Candidate B:
#     - tokenize example_row["candidate_B"], compute its p1, p2, bp, and BLEU-2 the same way
#   Compare both with sentence_bleu(candidate, [reference]) and print
#   (see worked target output for the format)


### Checkpoint 6.1: Connecting code to formula

1. Point to the exact line in your `modified_precision` where clipping happens. Explain in one sentence why removing the `min` would let a degenerate candidate cheat.
2. Your manual BP for candidate B is 0.3057, but sacrebleu reports 0.323 for the same pair. Neither is wrong. What is different? Hint: you split on whitespace; sacrebleu's default `13a` tokenizer also splits punctuation off words, so both length counts change.
3. Your manual score is BLEU-2 and sacrebleu's is smoothed BLEU-4, so the headline numbers differ too. List the three implementation choices you would have to pin down before comparing your BLEU numbers with anyone else's. This is exactly why the deck says a BLEU score quoted without its configuration is close to meaningless, and why sacrebleu exists.


## 7. Reflection and Stretch Goals

### 7.1 Reflection

Discuss with your group:

1. For the Cordwell rewriting project, is BLEU a reasonable primary metric? A reasonable **guardrail** metric? What is the difference?
2. You now know System B passes its input through 6 percent of the time and BLEU rewards it with perfect scores. Design one cheap automated check that would catch this in production before any human noticed.
3. How would you explain "System A scored 67.5 BLEU, System B scored 19.5" to a Cordwell product manager in two sentences, without using the words n-gram, precision, or penalty?

### 7.2 Stretch goals (optional, for fast finishers)

Solutions to both stretch goals are in the instructor solution notebook, distributed after the lab.

**Stretch 1: Build System C, the keyword stuffer.** Write `simulate_system_c(reference, rng)` that extracts the content words from the reference (say, words longer than 4 characters), shuffles them, and joins up to 40 of them into one long "sentence." Add a `candidate_C` column (use `random.Random(7)` for reproducibility), compute its corpus BLEU, and look at the component breakdown. Predict before you run it: what will its unigram precision be, and what will its 4-gram precision be? The answer is one of the most dramatic component splits you will ever see, and it explains precisely why BLEU multiplies four n-gram orders together instead of stopping at one.

**Stretch 2: A second opinion from chrF.** sacrebleu also ships `corpus_chrf`, a character n-gram F-score that is more forgiving of word reordering and morphology. Compute chrF for Systems A, B, and C and compare the **rankings** (not the absolute numbers) against BLEU's. One pair of systems swaps order between the two metrics. Find the swap and explain, in terms of what each metric counts, why it happens.


## Wrap-Up

You have now:

- Computed corpus-level BLEU with sacrebleu and decomposed scores into per-order precisions and the brevity penalty.
- Seen that System B's low score comes almost entirely from the brevity penalty while its surviving n-grams are actually fine, a diagnosis you could only make from the components.
- Measured sentence-level BLEU, seen its noise, and used score deltas to catch a system gaming the metric with verbatim passthrough.
- Rebuilt clipped precision and the brevity penalty by hand and reconciled your numbers with sacrebleu, including why tokenization changes the answer.

Next up: ROUGE for summarization-style evaluation, then semantic metrics, and eventually wiring all of these into an evaluation harness with experiment tracking so results like today's are logged, versioned, and comparable across runs.

Run the final cell to check your work.


In [ ]:
run_checks()
